# Internet Of Things: second challenge part two
 
## Authors
- Lorenzo Bardelli: 10831941
- Pietro Pizzoccheri (Team Leader): 10797420

## Part 2: Exercise (EQ1 & EQ2)

### Questions:
- Exercise Question 1 (EQ1):
Compute the total communication energy consumed by the battery-powered sensor
and actuator over 1 hour in:
(a) CoAP (direct communication)
(b) MQTT (via the gateway)
Provide numerical results in μJ (microjoule).
- Exercise Question 2 (EQ2):
Assume now that the actuator wakes up only once every 30 minutes.
Which protocol would you choose for long-term operation?
Justify your answer (max 300 characters), explicitly stating any protocol parameter
required (e.g., clean session, retain, observe, block-transfer modes etc...) and
performance metric you optimize.

In [3]:
# EQ1 computation
ETX_nJ = 50
ERX_nJ = 58
events_per_hour = 60 // 2  # one measurement every 2 minutes -> 30 events/hour

def bits(n_bytes: int) -> int:
    return 8 * n_bytes

# CoAP direct per event: sensor TX PUT + sensor RX ACK + actuator RX PUT + actuator TX ACK
coap_per_event_nJ = (
    bits(70) * ETX_nJ +
    bits(15) * ERX_nJ +
    bits(70) * ERX_nJ +
    bits(15) * ETX_nJ
)
eq1a_uJ = coap_per_event_nJ * events_per_hour / 1000.0

# MQTT per event:
# sensor TX PUBLISH + sensor RX PUBACK + actuator RX PUBLISH + actuator TX PUBACK
mqtt_per_event_nJ = (
    bits(75) * ETX_nJ +
    bits(50) * ERX_nJ +
    bits(75) * ERX_nJ +
    bits(50) * ETX_nJ
)

# one-time MQTT setup in the hour (devices start disconnected):
# sensor CONNECT/CONNACK + actuator CONNECT/CONNACK + actuator SUBSCRIBE/SUBACK
sensor_connect_once_nJ = bits(60) * ETX_nJ + bits(50) * ERX_nJ
actuator_connect_once_nJ = bits(60) * ETX_nJ + bits(50) * ERX_nJ
subscribe_once_nJ = bits(65) * ETX_nJ + bits(55) * ERX_nJ
mqtt_setup_once_nJ = sensor_connect_once_nJ + actuator_connect_once_nJ + subscribe_once_nJ

eq1b_uJ = (events_per_hour * mqtt_per_event_nJ + mqtt_setup_once_nJ) / 1000.0

print(f"EQ1a (CoAP): {eq1a_uJ:.1f} μJ")
print(f"EQ1b (MQTT): {eq1b_uJ:.2f} μJ")

EQ1a (CoAP): 2203.2 μJ
EQ1b (MQTT): 3385.92 μJ


### EQ1 answers
- logic: We calculate total communication energy in microjoules from TX/RX active bits over 1 hour.
- CoAP (direct): Sensor and actuator exchange CON PUT + ACK every 2 minutes.
- MQTT (via gateway): For EQ1, devices are assumed to stay connected during the 1-hour window (professor clarification), so CONNECT/CONNACK and SUBSCRIBE/SUBACK happen once at the beginning.

- EQ1a Answer: `2203.2 μJ`
- EQ1b Answer: `3385.92 μJ`

In [4]:
# EQ2
sensor_period_min = 2
actuator_wakeup_min = 30
samples_while_actuator_sleeping = actuator_wakeup_min // sensor_period_min

print(f"Sensor samples during one actuator sleep window: {samples_while_actuator_sleeping}")

Sensor samples during one actuator sleep window: 15


### EQ2 answer
- logic: Sensor samples every 2 minutes; direct CoAP wastes battery on retries while actuator sleeps.
- answer: MQTT with `retain=true` and `cleanSession=true` so the actuator reconnects and gets only the latest retained value.
- this minimizes traffic/energy by avoiding CoAP retransmissions and avoiding queued stale QoS1 messages.
- this optimizes actuator battery lifetime by minimizing communication energy.